In [1]:
import torch
import torch.nn as nn 
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import csv
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from torch.utils.data import Dataset, DataLoader

from transformers import CLIPModel, CLIPProcessor

/home/arnavw/Documents/amazon-ml-2025/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class CLIPPricePredictor(nn.Module):
    def __init__(self, model_name="openai/clip-vit-base-patch32"):
        super().__init__()
        # Load CLIP model
        self.clip = CLIPModel.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map="cuda"
        )
        
        # Freeze CLIP parameters
        for param in self.clip.parameters():
            param.requires_grad = False
        
        # Get CLIP projection dimension (typically 512 for base models)
        clip_dim = self.clip.config.projection_dim
        model_device = next(self.clip.parameters()).device
        
        # Regression head for multimodal features (concatenation fusion)
        self.regressor = nn.Sequential(
            nn.LayerNorm(clip_dim * 2),
            nn.Linear(clip_dim * 2, 256),
            nn.ReLU(),
            
            nn.Dropout(0.2),
            nn.Linear(256, 1),
            nn.ReLU()
        )
        
        self._init_weights()
        self.regressor.to(model_device, dtype=torch.float32)
    
    def _init_weights(self):
        for layer in self.regressor:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_normal_(layer.weight, gain=0.01)
                nn.init.zeros_(layer.bias)
    
    @torch.no_grad()
    def extract_features(self, pixel_values, input_ids, attention_mask):
        """
        Extract frozen multimodal embeddings from CLIP.
        Combines image and text features in the shared embedding space.
        """
        self.clip.eval()
        
        # Get CLIP outputs
        outputs = self.clip(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        
        # Extract normalized embeddings from CLIP's projection space
        image_embeds = outputs.image_embeds  # [batch_size, projection_dim]
        text_embeds = outputs.text_embeds    # [batch_size, projection_dim]
        
        # Concatenate image and text features (concatenation fusion)
        multimodal_features = torch.cat([image_embeds, text_embeds], dim=-1)
        
        # Clamp for stability
        return torch.clamp(multimodal_features.float(), min=-5, max=5)
    
    def forward(self, pixel_values, input_ids, attention_mask=None):
        """
        Forward pass for price prediction.
        
        Args:
            pixel_values: Preprocessed images [batch_size, 3, H, W]
            input_ids: Tokenized text [batch_size, seq_len]
            attention_mask: Attention mask for text [batch_size, seq_len]
        
        Returns:
            price_pred: Predicted prices [batch_size]
        """
        with torch.no_grad():
            features = self.extract_features(pixel_values, input_ids, attention_mask)
        
        price_pred = self.regressor(features)
        return price_pred.squeeze(-1)

In [3]:
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [4]:
# Create test dataset class
class TestDataset(Dataset):
    def __init__(self, csv_file, image_dir, processor):
        self.data = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.processor = processor
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        sample_id = row['sample_id']
        
        # Get image filename from image_link
        image_link = row['image_link']
        image_filename = image_link.split('/')[-1]  # Extract filename from URL
        image_path = os.path.join(self.image_dir, image_filename)
        
        try:
            # Load image
            image = Image.open(image_path).convert('RGB')
        except Exception as e:
            print(f"Error loading image {image_path}: {e}")
            # Create a black image as fallback
            image = Image.new('RGB', (224, 224), color='black')
        
        # Concatenate text fields
        text_fields = []
        if pd.notna(row['Item Name']):
            text_fields.append(str(row['Item Name']))
        if pd.notna(row['Product Description']):
            text_fields.append(str(row['Product Description']))
        if pd.notna(row['Bullet Points']):
            text_fields.append(str(row['Bullet Points']))
        
        text = " ".join(text_fields) if text_fields else "No description available"
        
        return {
            'sample_id': sample_id,
            'image': image,
            'text': text
        }

In [5]:
# Load the trained model
def load_model():
    # Initialize model
    model = CLIPPricePredictor()
    
    # Load trained weights
    model_path = "/home/arnavw/Documents/amazon-ml-2025/src/model_final_2.pth"
    checkpoint = torch.load(model_path, map_location='cuda' if torch.cuda.is_available() else 'cpu')
    
    # Load only the regressor weights (CLIP weights are frozen)
    if 'regressor' in checkpoint:
        model.regressor.load_state_dict(checkpoint['regressor'])
        print("Loaded regressor weights from checkpoint")
    elif 'model_state_dict' in checkpoint:
        # Try to load full model state dict
        model.load_state_dict(checkpoint['model_state_dict'], strict=False)
        print("Loaded model weights from checkpoint")
    else:
        # Assume checkpoint contains the full state dict
        model.load_state_dict(checkpoint, strict=False)
        print("Loaded weights directly from checkpoint")
    
    model.eval()
    return model

# Load model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
model = load_model().to(device)

`torch_dtype` is deprecated! Use `dtype` instead!


Using device: cuda
Loaded regressor weights from checkpoint


In [6]:
# Custom collate function to handle variable sized texts
def custom_collate_fn(batch):
    sample_ids = [item['sample_id'] for item in batch]
    images = [item['image'] for item in batch]
    texts = [item['text'] for item in batch]
    
    # Process all images and texts together for consistent padding
    inputs = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77
    )
    
    return {
        'sample_id': sample_ids,
        'pixel_values': inputs['pixel_values'],
        'input_ids': inputs['input_ids'],
        'attention_mask': inputs['attention_mask']
    }


In [ ]:
# Setup paths
csv_path = "/home/arnavw/Documents/amazon-ml-2025/dataset/test_split/part1.csv"
image_dir = "/home/arnavw/Documents/amazon-ml-2025/images/test_part1"
output_path = "/home/arnavw/Documents/amazon-ml-2025/test_predictions_1.csv"


# Create dataset and dataloader
test_dataset = TestDataset(csv_path, image_dir, processor)
test_dataloader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=0, collate_fn=custom_collate_fn)

print(f"Total samples to process: {len(test_dataset)}")
print(f"Number of batches: {len(test_dataloader)}")

Total samples to process: 25000
Number of batches: 3125


In [8]:
# Run inference
def run_inference(t_dataloader):
    predictions = []
    sample_ids = []
    
    model.eval()
    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(t_dataloader, desc="Running inference")):
            try:
                # Move to device
                pixel_values = batch['pixel_values'].to(device)
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                
                # Forward pass
                outputs = model(pixel_values, input_ids, attention_mask)
                
                # Store predictions
                predictions.extend(outputs.cpu().numpy().tolist())
                sample_ids.extend(batch['sample_id'])
                
            except Exception as e:
                print(f"Error processing batch {batch_idx}: {e}")
                # Add zeros for failed batches
                batch_size = len(batch['sample_id'])
                predictions.extend([0.0] * batch_size)
                sample_ids.extend(batch['sample_id'])
    
    return sample_ids, predictions

In [ ]:
# Run inference
print("Starting inference...")
sample_ids, predictions = run_inference()
print(f"Inference completed. Processed {len(sample_ids)} samples.")

In [16]:
# Save predictions to CSV
results_df = pd.DataFrame({
    'sample_id': sample_ids,
    'predicted_price': predictions
})

# Sort by sample_id to ensure consistent ordering
results_df = results_df.sort_values('sample_id')
results_df['predicted_price'] = np.expm1(results_df['predicted_price']).round(2)
# Save to CSV
results_df.to_csv(output_path, index=False)
print(f"Predictions saved to {output_path}")

# Display first few predictions
print("\nFirst 10 predictions:")
print(results_df.head(10))

print(f"\nSummary statistics:")
print(f"Total predictions: {len(results_df)}")
print(f"Mean predicted price: ${results_df['predicted_price'].mean():.2f}")
print(f"Min predicted price: ${results_df['predicted_price'].min():.2f}")
print(f"Max predicted price: ${results_df['predicted_price'].max():.2f}")

Predictions saved to /home/arnavw/Documents/amazon-ml-2025/test_predictions.csv

First 10 predictions:
       sample_id  predicted_price
4020           1            12.42
6852           9            43.79
10141         20            23.55
7442          28             8.02
5757          61            18.33
24766         86            29.60
1001         100            25.69
10138        102            19.68
7034         127             7.58
15321        134            18.08

Summary statistics:
Total predictions: 25000
Mean predicted price: $17.22
Min predicted price: $1.34
Max predicted price: $204.99


#### Part 2 

In [9]:
del sample_ids, predictions
import gc 
gc.collect()
torch.cuda.empty_cache()

NameError: name 'sample_ids' is not defined

In [10]:
csv_path = "/home/arnavw/Documents/amazon-ml-2025/dataset/test_split/part2.csv"
image_dir = "/home/arnavw/Documents/amazon-ml-2025/images/test_part2"
output_path = "/home/arnavw/Documents/amazon-ml-2025/test_predictions_2.csv"

test_dataset = TestDataset(csv_path, image_dir, processor)
test_dataloader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=custom_collate_fn)

print(f"Total samples to process: {len(test_dataset)}")
print(f"Number of batches: {len(test_dataloader)}")

Total samples to process: 25000
Number of batches: 3125


In [11]:
# Run inference
print("Starting inference...")
sample_ids, predictions = run_inference(test_dataloader)
print(f"Inference completed. Processed {len(sample_ids)} samples.")

Starting inference...


Running inference:  68%|██████▊   | 2130/3125 [13:37<05:22,  3.08it/s]

Error loading image /home/arnavw/Documents/amazon-ml-2025/images/test_part2/813CjSgHj0S.jpg: [Errno 2] No such file or directory: '/home/arnavw/Documents/amazon-ml-2025/images/test_part2/813CjSgHj0S.jpg'


Running inference: 100%|██████████| 3125/3125 [19:35<00:00,  2.66it/s]

Inference completed. Processed 25000 samples.


In [12]:
# Save predictions to CSV
results_df = pd.DataFrame({
    'sample_id': sample_ids,
    'predicted_price': predictions
})

# Sort by sample_id to ensure consistent ordering
results_df = results_df.sort_values('sample_id')
results_df['predicted_price'] = np.expm1(results_df['predicted_price']).round(2)
# Save to CSV
results_df.to_csv(output_path, index=False)
print(f"Predictions saved to {output_path}")

# Display first few predictions
print("\nFirst 10 predictions:")
print(results_df.head(10))

print(f"\nSummary statistics:")
print(f"Total predictions: {len(results_df)}")
print(f"Mean predicted price: ${results_df['predicted_price'].mean():.2f}")
print(f"Min predicted price: ${results_df['predicted_price'].min():.2f}")
print(f"Max predicted price: ${results_df['predicted_price'].max():.2f}")

Predictions saved to /home/arnavw/Documents/amazon-ml-2025/test_predictions_2.csv

First 10 predictions:
       sample_id  predicted_price
14491          3            24.64
13023         21             9.65
3586          23            25.40
7136          29            19.55
13710         48            11.71
18453         55            13.69
6736          57            13.86
13466         58            10.09
8104          62             9.26
7863          63            19.81

Summary statistics:
Total predictions: 25000
Mean predicted price: $17.21
Min predicted price: $1.29
Max predicted price: $198.41


#### Part 3

In [13]:
del sample_ids, predictions
import gc 
gc.collect()
torch.cuda.empty_cache()

In [14]:
csv_path = "/home/arnavw/Documents/amazon-ml-2025/dataset/test_split/part3.csv"
image_dir = "/home/arnavw/Documents/amazon-ml-2025/images/test_part3"
output_path = "/home/arnavw/Documents/amazon-ml-2025/test_predictions_3.csv"

test_dataset = TestDataset(csv_path, image_dir, processor)
test_dataloader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=custom_collate_fn)

print(f"Total samples to process: {len(test_dataset)}")
print(f"Number of batches: {len(test_dataloader)}")

Total samples to process: 25000
Number of batches: 3125


In [15]:
# Run inference
print("Starting inference...")
sample_ids, predictions = run_inference(test_dataloader)
print(f"Inference completed. Processed {len(sample_ids)} samples.")

Starting inference...


Running inference:  42%|████▏     | 1310/3125 [07:27<10:05,  3.00it/s]

Error loading image /home/arnavw/Documents/amazon-ml-2025/images/test_part3/41XNmOHw6iL.jpg: cannot identify image file '/home/arnavw/Documents/amazon-ml-2025/images/test_part3/41XNmOHw6iL.jpg'


Running inference: 100%|██████████| 3125/3125 [17:56<00:00,  2.90it/s]

Inference completed. Processed 25000 samples.


In [16]:
# Save predictions to CSV
results_df = pd.DataFrame({
    'sample_id': sample_ids,
    'predicted_price': predictions
})

# Sort by sample_id to ensure consistent ordering
results_df = results_df.sort_values('sample_id')
results_df['predicted_price'] = np.expm1(results_df['predicted_price']).round(2)
# Save to CSV
results_df.to_csv(output_path, index=False)
print(f"Predictions saved to {output_path}")

# Display first few predictions
print("\nFirst 10 predictions:")
print(results_df.head(10))

print(f"\nSummary statistics:")
print(f"Total predictions: {len(results_df)}")
print(f"Mean predicted price: ${results_df['predicted_price'].mean():.2f}")
print(f"Min predicted price: ${results_df['predicted_price'].min():.2f}")
print(f"Max predicted price: ${results_df['predicted_price'].max():.2f}")

Predictions saved to /home/arnavw/Documents/amazon-ml-2025/test_predictions_3.csv

First 10 predictions:
       sample_id  predicted_price
14597         19            17.54
5653          25            13.43
10198         27            15.80
24885         45            31.28
24023         50            17.36
15306         67             6.43
18443         71            16.12
18380         82             9.80
8310          99            21.10
18462        116            11.38

Summary statistics:
Total predictions: 25000
Mean predicted price: $17.16
Min predicted price: $1.34
Max predicted price: $209.13


### Finally Put all in one CSV File

In [17]:
test_df = pd.read_csv("/home/arnavw/Documents/amazon-ml-2025/dataset/test.csv")

# Read all prediction files
pred1_df = pd.read_csv("/home/arnavw/Documents/amazon-ml-2025/test_predictions_1.csv")
pred2_df = pd.read_csv("/home/arnavw/Documents/amazon-ml-2025/test_predictions_2.csv")
pred3_df = pd.read_csv("/home/arnavw/Documents/amazon-ml-2025/test_predictions_3.csv")

# Combine all predictions
all_predictions = pd.concat([pred1_df, pred2_df, pred3_df], ignore_index=True)

# Merge with original test.csv to maintain the same order
final_predictions = test_df[['sample_id']].merge(
    all_predictions, 
    on='sample_id', 
    how='left'
)

# Save the final predictions
final_output_path = "/home/arnavw/Documents/amazon-ml-2025/final_test_predictions.csv"
final_predictions.to_csv(final_output_path, index=False)

print(f"Final predictions saved to {final_output_path}")
print(f"Total samples: {len(final_predictions)}")
print(f"Samples with predictions: {final_predictions['predicted_price'].notna().sum()}")
print(f"Missing predictions: {final_predictions['predicted_price'].isna().sum()}")

# Display first few predictions
print("\nFirst 10 predictions:")
print(final_predictions.head(10))

# Display summary statistics
valid_predictions = final_predictions['predicted_price'].dropna()
if len(valid_predictions) > 0:
    print(f"\nSummary statistics:")
    print(f"Mean predicted price: ${valid_predictions.mean():.2f}")
    print(f"Min predicted price: ${valid_predictions.min():.2f}")
    print(f"Max predicted price: ${valid_predictions.max():.2f}")

Final predictions saved to /home/arnavw/Documents/amazon-ml-2025/final_test_predictions.csv
Total samples: 75000
Samples with predictions: 75000
Missing predictions: 0

First 10 predictions:
   sample_id  predicted_price
0     100179            12.64
1     245611            18.83
2     146263            18.29
3      95658             8.27
4      36806            19.05
5     148239             4.45
6      92659            11.16
7       3780            10.15
8     196940            19.92
9      20472             5.88

Summary statistics:
Mean predicted price: $17.20
Min predicted price: $1.29
Max predicted price: $209.13
